# Ingestión Manual de Datos (PDFs y Requisitos)
Este cuaderno permite ingestar directamente a la base de datos (PostgreSQL/pgvector) los requerimientos etiquetados del dataset de prueba y procesar la norma ISO 29148 desde su PDF, dividiéndola en fragmentos (chunks) de texto de tamaño configurado.

## 1. Configuración Inicial

In [1]:
import sys
import os
import time
import pandas as pd
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')

load_dotenv('.env')
sys.path.append(os.path.abspath('../backend'))

# Forzar el uso de localhost para LM Studio ya que el notebook corre fuera de Docker
os.environ["LOCAL_MODELS_API"] = "http://localhost:1234/v1/"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pdfplumber

from app.services.retriever.retriever_service import get_embeddings
from app.db.session import SessionLocal
from app.db.models import Document as DBDocument
from app.db.models import Requirement as DBRequirement

In [2]:
def index_documents(chunks: list[Document], batch_size: int = 10, delay: float = 2.0) -> int:
    if not chunks:
        return 0

    embeddings_model = get_embeddings()
    total_indexed = 0
    db = SessionLocal()
    
    try:
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i : i + batch_size]
            texts = [chunk.page_content for chunk in batch]
            
            try:
                embeddings = embeddings_model.embed_documents(texts)
                db_docs = []
                for chunk, emb in zip(batch, embeddings):
                    db_docs.append(
                        DBDocument(
                            content=chunk.page_content,
                            source=chunk.metadata.get("source_file", chunk.metadata.get("source", "unknown")),
                            page=chunk.metadata.get("page"),
                            chunk_index=chunk.metadata.get("chunk_index"),
                            embedding=emb,
                            metadata_=chunk.metadata
                        )
                    )
                db.add_all(db_docs)
                db.commit()
                total_indexed += len(batch)
            except Exception as e:
                db.rollback()
                time.sleep(delay * 3)
                try:
                    embeddings = embeddings_model.embed_documents(texts)
                    db_docs = []
                    for chunk, emb in zip(batch, embeddings):
                        db_docs.append(
                            DBDocument(
                                content=chunk.page_content,
                                source=chunk.metadata.get("source_file", chunk.metadata.get("source", "unknown")),
                                page=chunk.metadata.get("page"),
                                chunk_index=chunk.metadata.get("chunk_index"),
                                embedding=emb,
                                metadata_=chunk.metadata
                            )
                        )
                    db.add_all(db_docs)
                    db.commit()
                    total_indexed += len(batch)
                except Exception as e2:
                    db.rollback()
                    print(f"Error fatal indexando batch de chunks: {str(e2)}")
                    continue

            if i + batch_size < len(chunks):
                time.sleep(delay)
                
    finally:
        db.close()

    return total_indexed

def index_requirements(texts: list[str], source: str = "manual", source_name: str = "unknown", batch_size: int = 10, delay: float = 2.0) -> int:
    if not texts:
        return 0
        
    embeddings_model = get_embeddings()
    texts = [text for text in texts if text.strip()]
    if not texts:
        return 0
        
    db = SessionLocal()
    total_indexed = 0
    
    try:
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            
            try:
                embeddings = embeddings_model.embed_documents(batch)
                db_reqs = []
                for text, emb in zip(batch, embeddings):
                    db_reqs.append(
                        DBRequirement(
                            text=text,
                            source=source,
                            source_name=source_name,
                            embedding=emb,
                            metadata_={"source": source, "source_name": source_name}
                        )
                    )
                db.add_all(db_reqs)
                db.commit()
                total_indexed += len(db_reqs)
            except Exception as e:
                db.rollback()
                time.sleep(delay * 3)
                try:
                    embeddings = embeddings_model.embed_documents(batch)
                    db_reqs = []
                    for text, emb in zip(batch, embeddings):
                        db_reqs.append(
                            DBRequirement(
                                text=text,
                                source=source,
                                source_name=source_name,
                                embedding=emb,
                                metadata_={"source": source, "source_name": source_name}
                            )
                        )
                    db.add_all(db_reqs)
                    db.commit()
                    total_indexed += len(db_reqs)
                except Exception as e2:
                    db.rollback()
                    print(f"Error fatal indexando batch de requisitos: {str(e2)}")
                    continue

            if i + batch_size < len(texts):
                time.sleep(delay)
                
        return total_indexed
    finally:
        db.close()


## 2. Ingesta de Requerimientos desde CSV
Cada requerimiento se almacena como un único fragmento en la tabla `requirements`.

In [3]:
csv_path = '../data/raw/requirements.csv'

print("Cargando dataset de requerimientos...")
df_reqs = pd.read_csv(csv_path)

# Limpiamos nulos y obtenemos la lista de textos
req_texts = df_reqs['requirement'].dropna().tolist()
print(f"Total de requerimientos a indexar: {len(req_texts)}")

indexed_count = index_requirements(
    texts=req_texts,
    source="csv",
    source_name="requirements.csv"
)

print(f"Requerimientos indexados exitosamente: {indexed_count}")

Cargando dataset de requerimientos...
Total de requerimientos a indexar: 173
Requerimientos indexados exitosamente: 173


## 3. Preprocesamiento e Ingesta del PDF (Norma ISO)
Se lee el texto del PDF, se elimina ruido (espacios, saltos de página) y se divide en fragmentos de **400 caracteres con 100 de solapamiento**.

In [4]:
pdf_path = '../data/raw/iso_29148.pdf'

def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    return text

print("Extrayendo texto del PDF de la norma ISO...")
if os.path.exists(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)
    print(f"Caracteres extraídos: {len(raw_text)}")
    
    # Configurar el text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Crear chunks
    chunks = text_splitter.split_text(raw_text)
    print(f"Se generaron {len(chunks)} chunks.")
    
    # Convertir a objetos Document de LangChain
    documents = [
        Document(
            page_content=chunk, 
            metadata={"source": "pdf", "source_file": "iso_29148.pdf", "chunk_index": i}
        ) 
        for i, chunk in enumerate(chunks)
    ]
    
    # Indexar documentos en la base de datos vectorial
    print("Indexando chunks en la base de datos...")
    docs_indexed = index_documents(documents)
    print(f"Chunks indexados exitosamente: {docs_indexed}")
else:
    print(f"No se encontró el archivo: {pdf_path}")

Extrayendo texto del PDF de la norma ISO...
Caracteres extraídos: 327211
Se generaron 1080 chunks.
Indexando chunks en la base de datos...
Chunks indexados exitosamente: 1080
